In [1]:
import datasets
import os
import pandas as pd
import json
from create_datasets import get_label_set
import numpy as np

cwd = os.getcwd()

/home/marwas/polianna_work/polienv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
letter = "a"
htype = "mhead"
dsdct = datasets.load_from_disk(f"{cwd}/inputs/{letter}/{htype}_dsdcts/dsdct_r2")
label_lst = get_label_set(letter, htype)

In [14]:
entry = 6
lbls = ['labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time']
for i in range(len(ds[entry]['tokens'])):
    #print(f"{ds[entry]['tokens'][i]} {ds[entry]['ner_tags'][i]}")
    string = f"{ds[entry]['tokens'][i]} "
    for lbl in lbls:
        string+= f"{ds[entry][lbl][i]} "
    print(string)

it O O O O O 
shall O O O O O 
not O O O O O 
affect O O O O O 
the O O O O O 
validity O O O O O 
of O O O O O 
any O O O O O 
delegated O B O O O 
acts O I O O O 
already O O O O O 
in O O O O O 
force O O O O O 
. O O O O O 


start with the a dataset

In [3]:
letter = "a"
htype = "mhead"
ds = datasets.load_from_disk(f"{cwd}/inputs/{letter}/{htype}_ds")
label_lst = get_label_set(letter, htype)

get total spans&tokens, per-label spans&tokens, and ratios

In [4]:
total_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
total_spans = 0
total_labeled_tokens = 0
for row in ds:
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        total_counts[label]["spans"] += spans
        total_counts[label]["tokens"] += spans+innies
        #
        total_spans+= spans
        total_labeled_tokens+=spans+innies
print(f"Total Spans: {total_spans}; Total (Labeled) Tokens: {total_labeled_tokens}")
print(total_counts)
for label in list(total_counts):
    print(f"{label}: {round((total_counts[label]['spans']/total_spans)*100,2)}% all spans, {round((total_counts[label]['tokens']/total_labeled_tokens)*100,2)}% all (labeled) tokens")

Total Spans: 10706; Total (Labeled) Tokens: 31694
{'Actor': {'spans': 5381, 'tokens': 11838}, 'InstrumentType': {'spans': 3145, 'tokens': 7426}, 'Objective': {'spans': 942, 'tokens': 7991}, 'Resource': {'spans': 481, 'tokens': 1217}, 'Time': {'spans': 757, 'tokens': 3222}}
Actor: 50.26% all spans, 37.35% all (labeled) tokens
InstrumentType: 29.38% all spans, 23.43% all (labeled) tokens
Objective: 8.8% all spans, 25.21% all (labeled) tokens
Resource: 4.49% all spans, 3.84% all (labeled) tokens
Time: 7.07% all spans, 10.17% all (labeled) tokens


In [5]:
ds_sent = datasets.load_from_disk(f"{cwd}/inputs/{letter}/sent/{htype}_ds")
total_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
total_spans = 0
total_labeled_tokens = 0
for row in ds_sent:
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        total_counts[label]["spans"] += spans
        total_counts[label]["tokens"] += spans+innies
        #
        total_spans+= spans
        total_labeled_tokens+=spans+innies
print(f"Total Spans: {total_spans}; Total (Labeled) Tokens: {total_labeled_tokens}")
print(total_counts)
for label in list(total_counts):
    print(f"{label}: {round((total_counts[label]['spans']/total_spans)*100,2)}% all spans, {round((total_counts[label]['tokens']/total_labeled_tokens)*100,2)}% all (labeled) tokens")

Total Spans: 10706; Total (Labeled) Tokens: 31694
{'Actor': {'spans': 5381, 'tokens': 11838}, 'InstrumentType': {'spans': 3145, 'tokens': 7426}, 'Objective': {'spans': 942, 'tokens': 7991}, 'Resource': {'spans': 481, 'tokens': 1217}, 'Time': {'spans': 757, 'tokens': 3222}}
Actor: 50.26% all spans, 37.35% all (labeled) tokens
InstrumentType: 29.38% all spans, 23.43% all (labeled) tokens
Objective: 8.8% all spans, 25.21% all (labeled) tokens
Resource: 4.49% all spans, 3.84% all (labeled) tokens
Time: 7.07% all spans, 10.17% all (labeled) tokens


In [ ]:
#weights
wghts_a_1 = {}
#wghts_a_2 = {}
for lbl in list(total_counts):
    #print(f"{lbl}: {total_spans/(total_counts[lbl]['spans']*len(total_counts))}")
    wghts_a_1[lbl]= total_spans/(total_counts[lbl]['spans']*len(total_counts))
    #wghts_a_2[lbl]= 1-total_counts[lbl]['spans']/total_spans
wghts_a_1#, wghts_a_2

({'Actor': 0.39791860249024347,
  'InstrumentType': 0.6808267090620032,
  'Objective': 2.273036093418259,
  'Resource': 4.451559251559251,
  'Time': 2.828533685601057},
 {'Actor': 0.4973846441247899,
  'InstrumentType': 0.7062394918737156,
  'Objective': 0.9120119559125723,
  'Resource': 0.9550719222865682,
  'Time': 0.9292919858023538})

for each of the 412 articles, get label counts and total percentages for spans&tokens

In [12]:
tracking = {}
for row in ds:
    tracking[row['id']] = {label: {"spans":0,"tokens":0} for label in label_lst}
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        tracking[row['id']][label]["spans"] = spans
        tracking[row['id']][label]["tokens"] = spans+innies
        #
        tracking[row['id']][label]["span_pct"] = round((spans/total_counts[label]['spans'])*100,3)
        tracking[row['id']][label]["token_pct"] = round(((spans+innies)/total_counts[label]['tokens'])*100,3)

In [14]:
tracking_sent = {}
for row in ds_sent:
    tracking_sent[row['id']] = {label: {"spans":0,"tokens":0} for label in label_lst}
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        tracking_sent[row['id']][label]["spans"] = spans
        tracking_sent[row['id']][label]["tokens"] = spans+innies
        #
        tracking_sent[row['id']][label]["span_pct"] = round((spans/total_counts[label]['spans'])*100,3)
        tracking_sent[row['id']][label]["token_pct"] = round(((spans+innies)/total_counts[label]['tokens'])*100,3)

for each label, gathering the articles where they occur

In [15]:
arts_of_occurrence = {label:[] for label in label_lst}
for artid in list(tracking):
    for label in label_lst:
        spct = tracking[artid][label]["span_pct"]
        if spct>0:
            #arts_of_occurrence[label].append((artid, spct))
            arts_of_occurrence[label].append(artid)
for label in list(arts_of_occurrence):
    print(f"{label}: {len(arts_of_occurrence[label])}")

Actor: 371
InstrumentType: 312
Objective: 204
Resource: 135
Time: 246


In [17]:
sents_of_occurrence = {label:[] for label in label_lst}
for sentid in list(tracking_sent):
    for label in label_lst:
        spct = tracking_sent[sentid][label]["span_pct"]
        if spct>0:
            #sents_of_occurrence[label].append((sentid, spct))
            sents_of_occurrence[label].append(sentid)
for label in list(sents_of_occurrence):
    print(f"{label}: {len(sents_of_occurrence[label])}")

Actor: 2130
InstrumentType: 1444
Objective: 460
Resource: 275
Time: 534


find a way to dynamically create a threshold of inclusion of minority classes

In [20]:
lbl_arts_occ = []
min_obj = (None,100)
max_obj = (None,0)
for label in list(arts_of_occurrence):
    val = round((len(arts_of_occurrence[label])/412)*100,2)
    lbl_arts_occ.append((label,val))
    print(f"{label}: {val}")
    if val<min_obj[1]:
        min_obj = (label,val)
    if val>max_obj[1]:
        max_obj = (label,val)
#for label in list(arts_of_occurrence):
#    if len(arts_of_occurrence[label])/412<0.5:
#        minority_labels.append(label)
vals = [entry[1] for entry in lbl_arts_occ]
thresh = np.quantile(vals, .333)
print(f"Thresh: {thresh}")
minority_labels = []
for entry in lbl_arts_occ:
    if entry[1]<thresh:
        minority_labels.append(entry[0])
print(minority_labels)
print(min_obj, max_obj)
min_label = min_obj[0]
max_label = max_obj[0]

Actor: 90.05
InstrumentType: 75.73
Objective: 49.51
Resource: 32.77
Time: 59.71
Thresh: 52.8964
['Objective', 'Resource']
('Resource', 32.77) ('Actor', 90.05)


In [57]:
lbl_sents_occ = []
min_obj = (None,100)
max_obj = (None,0)
for label in list(sents_of_occurrence):
    val = round((len(sents_of_occurrence[label])/412)*100,2)
    lbl_sents_occ.append((label,val))
    print(f"{label}: {val}")
    if val<min_obj[1]:
        min_obj = (label,val)
    if val>max_obj[1]:
        max_obj = (label,val)
#for label in list(sents_of_occurrence):
#    if len(sents_of_occurrence[label])/412<0.5:
#        minority_labels.append(label)
vals = [entry[1] for entry in lbl_sents_occ]
thresh = np.quantile(vals, .75)
print(f"Thresh: {thresh}")
minority_labels = []
for entry in lbl_sents_occ:
    if entry[1]<thresh:
        minority_labels.append(entry[0])
print(minority_labels)
print(min_obj, max_obj)
min_label = min_obj[0]
max_label = max_obj[0]

Actor: 516.99
InstrumentType: 350.49
Objective: 111.65
Resource: 66.75
Time: 129.61
Thresh: 350.49
['Objective', 'Resource', 'Time']
('Resource', 66.75) ('Actor', 516.99)


building a new list of articles based on lowest minority classes

In [49]:
arts_of_interest = []
for label in minority_labels:
    arts_of_interest.extend(arts_of_occurrence[label])
arts_of_interest = list(set(arts_of_interest))
print(len(arts_of_interest))

368


In [58]:
sents_of_interest = []
for label in minority_labels:
    sents_of_interest.extend(sents_of_occurrence[label])
print(len(sents_of_interest))
sents_of_interest = list(set(sents_of_interest))
print(len(sents_of_interest))

1269
1039


some method of sorting/reducing the number of articles to oversample from
for now, sorting descending by pct of most minority label span
then taking the first half of that list

In [51]:
aoi_dict = dict()
for key in arts_of_interest:
    aoi_dict[key] = tracking[key]
# sort by lowest minority label
#arts_inorder = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
arts_dscmin = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
arts_ascmax = sorted(aoi_dict, key=lambda x:aoi_dict[x][max_label]['span_pct'], reverse=False)
#aoi_dict_sorted = {key: aoi_dict[key] for key in arts_inorder}
aoi_dict_dscmin = {key: aoi_dict[key] for key in arts_dscmin}
aoi_dict_ascmax = {key: aoi_dict[key] for key in arts_ascmax}
# may need to find a way of reducing the number of articles dynamically?
# for now just take the first half
#overs_arts = arts_inorder[:int(len(arts_inorder)/2)]
overs_dscmin = arts_dscmin[:int(len(arts_dscmin)/2)]
overs_ascmax = arts_ascmax[:int(len(arts_ascmax)/2)]
overs_arts = list(set(overs_dscmin+overs_ascmax))

In [59]:
soi_dict = dict()
for key in sents_of_interest:
    soi_dict[key] = tracking_sent[key]
overs_sents = list(set(list(soi_dict)))
len(overs_sents)

1039

make a new dataset, the oversampling dataset, using the new articles
then concatenate them

In [28]:
osds = ds.filter(lambda example: example["id"] in overs_arts)
ovs_ds = datasets.concatenate_datasets([ds, osds])
ovs_ds

Filter: 100%|██████████| 412/412 [00:00<00:00, 823.94 examples/s]


Dataset({
    features: ['id', 'text', 'tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time'],
    num_rows: 603
})

In [64]:
osds_s = ds_sent.filter(lambda example: example["id"] in overs_sents)
ovs_ds_s = datasets.concatenate_datasets([ds_sent, osds_s])
ovs_ds_s

Dataset({
    features: ['tokens', 'labels_Actor', 'labels_InstrumentType', 'labels_Objective', 'labels_Resource', 'labels_Time', 'id'],
    num_rows: 3798
})

In [62]:
new_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
new_spans = 0
new_labeled_tokens = 0
for row in ovs_ds:
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        new_counts[label]["spans"] += spans
        new_counts[label]["tokens"] += spans+innies
        #
        new_spans+= spans
        new_labeled_tokens+=spans+innies
print(f"Total Spans: {new_spans}; Total (Labeled) Tokens: {new_labeled_tokens}")
print(new_counts)
for label in list(new_counts):
    print(f"{label}: {round((new_counts[label]['spans']/new_spans)*100,2)}% all spans, {round((new_counts[label]['tokens']/new_labeled_tokens)*100,2)}% all (labeled) tokens")

Total Spans: 17158; Total (Labeled) Tokens: 51868
{'Actor': {'spans': 8360, 'tokens': 18688}, 'InstrumentType': {'spans': 5035, 'tokens': 12086}, 'Objective': {'spans': 1657, 'tokens': 13996}, 'Resource': {'spans': 953, 'tokens': 2415}, 'Time': {'spans': 1153, 'tokens': 4683}}
Actor: 48.72% all spans, 36.03% all (labeled) tokens
InstrumentType: 29.34% all spans, 23.3% all (labeled) tokens
Objective: 9.66% all spans, 26.98% all (labeled) tokens
Resource: 5.55% all spans, 4.66% all (labeled) tokens
Time: 6.72% all spans, 9.03% all (labeled) tokens


In [65]:
new_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
new_spans = 0
new_labeled_tokens = 0
for row in ovs_ds_s:
    for label in label_lst:
        spans = row[f"labels_{label}"].count("B")
        innies = row[f"labels_{label}"].count("I")
        new_counts[label]["spans"] += spans
        new_counts[label]["tokens"] += spans+innies
        #
        new_spans+= spans
        new_labeled_tokens+=spans+innies
print(f"Total Spans: {new_spans}; Total (Labeled) Tokens: {new_labeled_tokens}")
print(new_counts)
for label in list(new_counts):
    print(f"{label}: {round((new_counts[label]['spans']/new_spans)*100,2)}% all spans, {round((new_counts[label]['tokens']/new_labeled_tokens)*100,2)}% all (labeled) tokens")

Total Spans: 16922; Total (Labeled) Tokens: 53256
{'Actor': {'spans': 7850, 'tokens': 17090}, 'InstrumentType': {'spans': 4712, 'tokens': 11306}, 'Objective': {'spans': 1884, 'tokens': 15982}, 'Resource': {'spans': 962, 'tokens': 2434}, 'Time': {'spans': 1514, 'tokens': 6444}}
Actor: 46.39% all spans, 32.09% all (labeled) tokens
InstrumentType: 27.85% all spans, 21.23% all (labeled) tokens
Objective: 11.13% all spans, 30.01% all (labeled) tokens
Resource: 5.68% all spans, 4.57% all (labeled) tokens
Time: 8.95% all spans, 12.1% all (labeled) tokens


In [67]:
def get_overall_counts(ds, label_lst):
    # initialize values
    total_counts = {label: {"spans":0,"tokens":0} for label in label_lst}
    total_spans = 0
    total_labeled_tokens = 0
    # get counts for label spans and tokens
    for row in ds:
        for label in label_lst:
            spans = row[f"labels_{label}"].count("B")
            innies = row[f"labels_{label}"].count("I")
            total_counts[label]["spans"] += spans
            total_counts[label]["tokens"] += spans+innies
            total_spans+= spans
            total_labeled_tokens+=spans+innies
    total_counts["Overall"] = {"spans":total_spans,"tokens":total_labeled_tokens}
    for label in list(total_counts):
        print(f"{label}: {round((total_counts[label]['spans']/total_spans)*100,2)}% all spans ({total_counts[label]['spans']})")
    return total_counts

def article_label_coverage(ds, label_lst, total_counts):
    tracking = {}
    for row in ds:
        tracking[row['id']] = {label: {"spans":0,"tokens":0} for label in label_lst}
        for label in label_lst:
            spans = row[f"labels_{label}"].count("B")
            innies = row[f"labels_{label}"].count("I")
            tracking[row['id']][label]["spans"] = spans
            tracking[row['id']][label]["tokens"] = spans+innies
            #
            tracking[row['id']][label]["span_pct"] = round((spans/total_counts[label]['spans'])*100,3)
            tracking[row['id']][label]["token_pct"] = round(((spans+innies)/total_counts[label]['tokens'])*100,3)
    return tracking

def get_art_lists_for_lbls(art_lbl_tracking, label_lst):
    arts_of_occurrence = {label:[] for label in label_lst}
    for artid in list(art_lbl_tracking):
        for label in label_lst:
            spct = art_lbl_tracking[artid][label]["span_pct"]
            if spct>0:
                arts_of_occurrence[label].append(artid)
    return arts_of_occurrence

def compare_art_lbl_occ(arts_of_occurrence, quant=0.333):
    lbl_arts_occ = []
    min_obj = (None,100)
    max_obj = (None,0)
    for label in list(arts_of_occurrence):
        val = round((len(arts_of_occurrence[label])/412)*100,2)
        lbl_arts_occ.append((label,val))
        #print(f"{label}: {val}")
        # get min and max label names
        if val<min_obj[1]:
            min_obj = (label,val)
        if val>max_obj[1]:
            max_obj = (label,val)
    # get threshold by finding quantile value
    vals = [entry[1] for entry in lbl_arts_occ]
    thresh = np.quantile(vals, quant)
    # determine minority labels
    minority_labels = []
    for entry in lbl_arts_occ:
        if entry[1]<thresh:
            minority_labels.append(entry[0])
    min_label = min_obj[0]
    max_label = max_obj[0]
    arts_of_interest = []
    for label in minority_labels:
        arts_of_interest.extend(arts_of_occurrence[label])
    arts_of_interest = list(set(arts_of_interest))
    print("Minority Labels:",minority_labels)
    return arts_of_interest, minority_labels, min_label, max_label

def compare_overall_lbl_occ(total_counts,arts_of_occurrence, quant=0.333):
    lbl_arts_occ = []
    min_obj = (None,100)
    max_obj = (None,0)
    for label in list(total_counts):
        val = round((total_counts[label]['spans']/total_counts['Overall']['spans'])*100,2)
        lbl_arts_occ.append((label,val))
        if val<min_obj[1]:
            min_obj = (label,val)
        if val>max_obj[1]:
            max_obj = (label,val)
    vals = [entry[1] for entry in lbl_arts_occ]
    thresh = np.quantile(vals, quant)
    minority_labels = []
    for entry in lbl_arts_occ:
        if entry[1]<thresh:
            minority_labels.append(entry[0])
    min_label = min_obj[0]
    max_label = max_obj[0]
    arts_of_interest = []
    for label in minority_labels:
        arts_of_interest.extend(arts_of_occurrence[label])
    arts_of_interest = list(set(arts_of_interest))
    print("Minority Labels:",minority_labels)
    return arts_of_interest, minority_labels, min_label, max_label

def refine_arts_of_int(method, art_lbl_tracking, arts_of_interest, minority_labels, min_label, max_label):
    aoi_dict = dict()
    for key in arts_of_interest:
        aoi_dict[key] = art_lbl_tracking[key]
    if method=="minority":# sort by lowest minority label
        arts_inorder = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
        #aoi_dict_sorted = {key: aoi_dict[key] for key in arts_inorder}
        overs_arts = arts_inorder[:int(len(arts_inorder)/2)]
    elif method=="dscmin+ascmax":
        arts_dscmin = sorted(aoi_dict, key=lambda x:aoi_dict[x][min_label]['span_pct'], reverse=True)
        arts_ascmax = sorted(aoi_dict, key=lambda x:aoi_dict[x][max_label]['span_pct'], reverse=False)
        #aoi_dict_dscmin = {key: aoi_dict[key] for key in arts_dscmin}
        #aoi_dict_ascmax = {key: aoi_dict[key] for key in arts_ascmax}
        # may need to find a way of reducing the number of articles dynamically?
        # for now just take the first half
        overs_dscmin = arts_dscmin[:int(len(arts_dscmin)/2)]
        overs_ascmax = arts_ascmax[:int(len(arts_ascmax)/2)]
        overs_arts = list(set(overs_dscmin+overs_ascmax))
    elif method=="minorities":
        tple_coll = {label:[] for label in minority_labels}
        art_coll = {label:[] for label in minority_labels}
        overs_arts = []
        for label in minority_labels:
            tplelst = sorted(aoi_dict, key=lambda x:aoi_dict[x][label]['span_pct'], reverse=True)
            tple_coll[label] = tplelst
            art_coll[label] = tplelst[:int(len(tplelst)/len(minority_labels))]
            overs_arts.extend(art_coll[label])
        overs_arts = list(set(overs_arts))
    return overs_arts

def create_new_osds(ds, overs_arts):
    osds = ds.filter(lambda example: example["id"] in overs_arts)
    ovs_ds = datasets.concatenate_datasets([ds, osds])
    return ovs_ds

In [73]:
letter = "a"
htype = "mhead"
ds = datasets.load_from_disk(f"{cwd}/inputs/{letter}/{htype}_ds")
label_lst = get_label_set(letter, htype)
total_counts = get_overall_counts(ds, label_lst)
art_lbl_tracking = article_label_coverage(ds, label_lst, total_counts)
arts_of_occurrence = get_art_lists_for_lbls(art_lbl_tracking, label_lst)
#arts_of_interest, minority_labels, min_label, max_label = compare_art_lbl_occ(arts_of_occurrence, quant=0.3)
arts_of_interest, minority_labels, min_label, max_label = compare_overall_lbl_occ(total_counts,arts_of_occurrence, quant=0.333)
overs_arts = refine_arts_of_int("minorities", art_lbl_tracking, arts_of_interest, minority_labels, min_label, max_label)
osds = create_new_osds(ds, overs_arts)
print("New")
new_counts = get_overall_counts(osds, label_lst)

Actor: 50.26% all spans (5381)
InstrumentType: 29.38% all spans (3145)
Objective: 8.8% all spans (942)
Resource: 4.49% all spans (481)
Time: 7.07% all spans (757)
Overall: 100.0% all spans (10706)
Minority Labels: ['Resource', 'Time']


Filter: 100%|██████████| 412/412 [00:00<00:00, 882.33 examples/s]


New
Actor: 49.41% all spans (9440)
InstrumentType: 29.11% all spans (5561)
Objective: 8.87% all spans (1694)
Resource: 5.04% all spans (962)
Time: 7.58% all spans (1448)
Overall: 100.0% all spans (19105)


In [79]:
letter = "a"
htype = "mhead"
ds_sent = datasets.load_from_disk(f"{cwd}/inputs/{letter}/sent/{htype}_ds")
print(len(ds_sent))
label_lst = get_label_set(letter, htype)
total_counts = get_overall_counts(ds_sent, label_lst)
sent_lbl_tracking = article_label_coverage(ds_sent, label_lst, total_counts)
sents_of_occurrence = get_art_lists_for_lbls(sent_lbl_tracking, label_lst)
#sents_of_interest, minority_labels, min_label, max_label = compare_art_lbl_occ(arts_of_occurrence, quant=0.3)
sents_of_interest, minority_labels, min_label, max_label = compare_overall_lbl_occ(total_counts,sents_of_occurrence, quant=0.6)
overs_sent = list(set([key for key in list(sents_of_interest)]))
osds_sent = create_new_osds(ds_sent, overs_sent)
print("New")
new_counts = get_overall_counts(osds_sent, label_lst)
print(len(sents_of_interest), len(osds_sent))

2759
Actor: 50.26% all spans (5381)
InstrumentType: 29.38% all spans (3145)
Objective: 8.8% all spans (942)
Resource: 4.49% all spans (481)
Time: 7.07% all spans (757)
Overall: 100.0% all spans (10706)
Minority Labels: ['Objective', 'Resource', 'Time']


Filter: 100%|██████████| 2759/2759 [00:00<00:00, 5428.30 examples/s]


New
Actor: 46.39% all spans (7850)
InstrumentType: 27.85% all spans (4712)
Objective: 11.13% all spans (1884)
Resource: 5.68% all spans (962)
Time: 8.95% all spans (1514)
Overall: 100.0% all spans (16922)
1039 3798


In [40]:
import torch
from collections import Counter
def calculate_tag_weights(dataset, head_lst):
    weights_dct = {}
    for head in head_lst:
        all_labels = [label for art in dataset[f'labels_{head}'] for label in art]
        counts = {"O":0,"B":0,"I":0}
        cntr = dict(Counter(all_labels))
        for key in list(cntr):
            counts[key] = cntr[key]
        # Calculate inverse frequency: total_count / (num_classes * class_count)
        total_tokens = 0
        for lbl in list(counts):
            total_tokens+= counts[lbl]
            if counts[lbl]<1:
                counts[lbl]=1# To avoid division by zero if a head has NO labels in a subset
        # weights = total / (n_classes * class_counts)
        counts_lst = [counts[key] for key in ["O","B","I"]]
        head_weights = [total_tokens / (3.0 * count) for count in counts_lst]
        # Normalize so the 'O' tag weight is always 1.0 
        norm_head_weights = [weight / head_weights[0] for weight in head_weights]
        weights_dct[head] = head_weights
        #print(f"Weights for {head}: O={norm_head_weights[0]:.2f}, B={norm_head_weights[1]:.2f}, I={norm_head_weights[2]:.2f}")   
    return weights_dct

calculate_tag_weights(dsdct['train'],label_lst)

{'Actor': [0.35951239856363276, 10.318421586528707, 8.227695543152956],
 'InstrumentType': [0.3478143421826488,
  18.209631220909415,
  14.288383199887624],
 'Objective': [0.3486619981421878, 66.74475065616798, 8.553565422132525],
 'Resource': [0.3357727602825642, 106.62368972746332, 80.53760886777513],
 'Time': [0.3394842938577169, 80.53760886777513, 23.8441162681669]}

In [38]:
def calculate_head_weights(dataset, head_lst):
    entities_dct = {}
    for head in head_lst:
        all_labels = [label for art in dataset[f'labels_{head}'] for label in art]
        cntr = dict(Counter(all_labels))
        try:
            entities_dct[head] = cntr["B"]
        except:
            entities_dct[head] = 0
    # inverse frequency: total_count / (num_classes * class_count)
    total_entities = 0
    for head in head_lst:
        total_entities+= entities_dct[head]
        if entities_dct[head]<1:
            entities_dct[head]=1# To avoid division by zero if a head has NO labels in a subset
    # weights = total / (n_classes * class_counts)
    counts_lst = [entities_dct[key] for key in head_lst]
    head_weights = [total_entities / (3.0 * count) for count in counts_lst]
    # Normalize so the 'O' tag weight is always 1.0 
    #norm_head_weights = [weight / head_weights[0] for weight in head_weights]
    weights_dct = {}
    for i, head in enumerate(head_lst):
        weights_dct[head] = torch.tensor(head_weights[i])
    return weights_dct

calculate_head_weights(dsdct['train'],label_lst)


{'Actor': tensor(0.6487),
 'InstrumentType': tensor(1.1448),
 'Objective': tensor(4.1962),
 'Resource': tensor(6.7034),
 'Time': tensor(5.0633)}